In [ ]:
# [demo ELC] basics
basic <- list(
  title='Demo of Equal Loudness Curve',
  author='Andy',
  date='Dec 2024',
  ref=c('https://cdn.standards.iteh.ai/samples/83117/6afa5bd94e0e4f32812c28c3b0a7b8ac/ISO-226-2023.pdf')
)

In [ ]:
# [demo ELC] load parameter table
library(readr)
raw_param_table <- '
freq,alpha_f,L_U,T_f
20,0.635,-31.5,78.1
25,0.602,-27.2,68.7
31.5,0.569,-23.1,59.5
40,0.537,-19.3,51.1
50,0.509,-16.1,44.0
63,0.482,-13.1,37.5
80,0.456,-10.4,31.5
100,0.433,-8.2,26.5
125,0.412,-6.3,22.1
160,0.391,-4.6,17.9
200,0.373,-3.2,14.4
250,0.357,-2.1,11.4
315,0.343,-1.2,8.6
400,0.330,-0.5,6.2
500,0.320,0.0,4.4
630,0.311,0.4,3.0
800,0.303,0.5,2.2
1000,0.300,0.0,2.4
1250,0.295,-2.7,3.5
1600,0.292,-4.2,1.7
2000,0.290,-1.2,-1.3
2500,0.290,1.4,-4.2
3150,0.289,2.3,-6.0
4000,0.289,1.0,-5.4
5000,0.289,-2.3,-1.5
6300,0.293,-7.2,6.0
8000,0.303,-11.2,12.6
10000,0.323,-10.9,13.9
12500,0.354,-3.5,12.3
'
dat <- read_csv(raw_param_table, col_types='dddd')
dat <- lapply(dat, function(col) col)

In [ ]:
# [demo ELC] arrange data
index_by_numeric_key <- function(S, k) { return(S[[as.character(k)]]) }
freq <- dat[['freq']]
alpha_f_ <- setNames(dat[['alpha_f']], freq)
alpha_f <- function(freq) { return(index_by_numeric_key(alpha_f_, freq)) }
L_U_ <- setNames(dat[['L_U']], freq)
L_U <- function(freq) { return(index_by_numeric_key(L_U_, freq)) }
T_f_ <- setNames(dat[['T_f']], freq)
T_f <- function(freq) { return(index_by_numeric_key(T_f_, freq)) }

In [ ]:
# [demo ELC] params for plot
library(scales)
Phon <- seq(from = 0, to = 140, by = 5)
breaks <- list(x=breaks_log(n=length(freq)), y=seq(0, 150, by=10))

In [ ]:
# [demo ELC] print basics
print(basic)

In [ ]:
# [demo ELC] plot equal loudness curve
library(ggplot2)
library(tidyr)

# Define functions for loudness calculations
L_f <- function(L_N, alpha_f, L_U, T_f) {
  return(10 / alpha_f * log10(4e-10 ^ (0.3 - alpha_f) * (10 ^ (0.03 * L_N) - 10 ^ 0.072) + 10 ^ (alpha_f * (T_f + L_U) / 10)) - L_U)
}

# Initialize a data frame to store results
results <- data.frame(freq = freq)

# Calculate loudness for each Phon value and store in the results data frame
for (p in Phon) {
  results[[as.character(p)]] <- sapply(freq, function(f) L_f(p, alpha_f(f), L_U(f), T_f(f)))
}

# for compatibility
reshaped_results <- as.list(results)

# Reshape the data for ggplot
results_long <- pivot_longer(results, cols = -freq, names_to = "Phon", values_to = "Loudness")

# Convert Phon to a factor for better color mapping
results_long[['Phon']] <- factor(results_long[['Phon']], levels=as.character(Phon))

# Plotting
chart <- ggplot(results_long, aes(x = `freq`, y = `Loudness`, color = `Phon`)) +
  geom_line() +
  scale_x_log10(breaks=breaks[['x']]) +
  scale_y_continuous(breaks=breaks[['y']])
  labs(title = "Equal Loudness Curves (ISO 226:2023)", x = "Frequency (Hz)", y = "Loudness Level (dB)", color = "Phon") +
  theme_minimal()
print(chart)

In [ ]:
# [demo ELC] example settings
param_seed <- 0
examples <- 10
min_dB <- 0
max_dB <- 150
param_digits <- 1

In [ ]:
# [demo ELC] examples of loudness units
library(stringr)

L_N <- function(L_f, alpha_f, L_U, T_f) {
  return(100 / 3 * log10((10 ^ (alpha_f * (L_f + L_U) / 10) - 10 ^ (alpha_f * (T_f + L_U) / 10)) / (4e-10 ^ (0.3 - alpha_f)) + 10 ^ 0.072))
}

set.seed(param_seed)
sound <- cbind(
  sample(freq, size=examples, replace=TRUE),
  round(runif(examples, min=min_dB, max=max_dB), digits=param_digits)
)

for (row in 1:nrow(sound)) {
  f <- sound[row, 1]
  p <- sound[row, 2]
  idx <- which(freq == f)
  print(stringr::str_interp("A sound (${f} Hz, ${p} dB), is ${round(p + 40 - reshaped_results[['40']][[idx]], digits=param_digits)} dBA, ${round(p + 70 - reshaped_results[['70']][[idx]], digits=param_digits)} dBB, ${round(p + 100 - reshaped_results[['100']][[idx]], digits=param_digits)} dBC, and ${round(L_N(p, alpha_f(f), L_U(f), T_f(f)), digits=param_digits)} Phon."))
}